# Entendimento inicial do dataset de voos

Este notebook explora o CSV derivado e a base de modelagem do Voo Regular Ativo (VRA) da ANAC. Ele não treina modelos. O objetivo é verificar estrutura, cobertura, qualidade, distribuição das seis faixas de atraso e outliers antes de definir as variáveis preditoras.

Para gerar a base local, execute primeiro o notebook `00_preparar_dados.ipynb`. O CSV é local e não é versionado.

### Importar as bibliotecas

Importamos as bibliotecas usadas para manipulação dos dados e visualização dos resultados.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

### Configurar a exibição das tabelas

Aumentamos a quantidade de colunas e linhas exibidas pelo pandas para facilitar a inspeção exploratória. Essa configuração não altera os dados.

In [ ]:
pd.set_option('display.max_columns', 100)
pd.set_option('display.max_rows', 100)

### Localizar a raiz do projeto

O notebook pode ser aberto a partir da raiz do projeto ou da pasta `notebooks`. Procuramos a pasta `data` subindo pelos diretórios para tornar os caminhos independentes do diretório de abertura.

In [ ]:
ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / 'data').exists():
    ROOT = ROOT.parent
print('Raiz do projeto:', ROOT)

### Localizar e validar as bases

Definimos os caminhos do dataset derivado e da base de modelagem e verificamos se o notebook `00_preparar_dados.ipynb` já foi executado.

In [ ]:
DATA_PATH = ROOT / 'data' / 'voos_vra_derivados.csv'
MODELAGEM_PATH = ROOT / 'data' / 'modelagem_faixas_atraso.csv'
for path in [DATA_PATH, MODELAGEM_PATH]:
    if not path.exists():
        raise FileNotFoundError(f'Arquivo não encontrado: {path}. Execute o notebook 00 primeiro.')
DATA_PATH, MODELAGEM_PATH

### Carregamento do dataset

Agora lemos o CSV derivado e a base de modelagem. Os quatro campos de data e hora do dataset derivado são convertidos para o tipo de data do pandas. A base de modelagem já contém `faixa_atraso` e as variáveis temporais usadas posteriormente.

In [ ]:
date_columns = ['partida_prevista', 'partida_real', 'chegada_prevista', 'chegada_real']
df = pd.read_csv(DATA_PATH, parse_dates=date_columns, low_memory=False)
modelagem = pd.read_csv(MODELAGEM_PATH, low_memory=False)
print('Dataset derivado:', df.shape)
print('Base de modelagem:', modelagem.shape)
df.head(3)

## 1. Exploração rápida para iniciantes

Depois de carregar uma base, é útil fazer algumas perguntas simples: quantas linhas e colunas existem, como são os primeiros registros, quais tipos de dados foram reconhecidos e quais são as principais estatísticas numéricas. Estas operações são descritivas: ajudam a entender a base antes de formular modelos.

### Tamanho e primeiras linhas

shape informa o número de linhas e colunas. head() mostra os primeiros registros para conferirmos se os dados foram carregados como esperado.

In [ ]:
print('Linhas e colunas:', df.shape)
df.head()

### Tipos das colunas e memória usada

info() resume o tipo de cada coluna, a quantidade de valores preenchidos e o uso aproximado de memória. É uma verificação importante para descobrir, por exemplo, se uma data foi lida como data ou como texto.

In [ ]:
df.info()

### Estatísticas numéricas com describe()

describe() calcula automaticamente estatísticas das colunas numéricas: quantidade (count), média (mean), desvio padrão (std), mínimo, quartis e máximo. O primeiro quartil é 25%, a mediana é 50% e o terceiro quartil é 75%.

In [14]:
df.describe().T

,count,mean,min,25%,50%,75%,max,std
numero_assentos,1005808.0,166.068101,0.0,136.0,180.0,186.0,515.0,66.987646
partida_prevista,977609,2025-07-02 06:53:52.006743,2025-01-01 00:00:00,2025-04-01 04:00:00,2025-07-03 07:30:00,2025-10-01 17:45:00,2026-01-02 09:05:00,NaN
partida_real,978092,2025-07-02 22:43:30.194337,2024-12-31 23:47:00,2025-04-01 17:38:00,2025-07-04 05:56:00,2025-10-02 17:24:00,2026-01-01 19:05:00,NaN
chegada_prevista,977609,2025-07-02 09:36:15.256652,2025-01-01 01:40:00,2025-04-01 07:10:00,2025-07-03 09:40:00,2025-10-01 20:05:00,2026-01-02 14:50:00,NaN
chegada_real,978092,2025-07-03 01:19:49.221320,2025-01-01 01:23:00,2025-04-01 19:46:00,2025-07-04 08:22:30,2025-10-02 19:34:00,2026-01-01 21:52:00,NaN
atraso_partida_min,949893.0,7.623539,-2890.0,-7.0,-2.0,8.0,44635.0,78.72343
atraso_chegada_min,949893.0,3.579488,-2984.0,-13.0,-5.0,7.0,44625.0,79.618394
linha_origem,1005808.0,41972.429655,1.0,20955.0,41909.0,62863.25,89616.0,24306.482207


### Média, mediana e outras medidas explicitamente

Também podemos pedir as medidas diretamente com agg(). A média pode ser influenciada por atrasos muito grandes; por isso comparamos com a mediana, que representa melhor o valor central quando há outliers.

In [ ]:
colunas_numericas = ['atraso_partida_min', 'atraso_chegada_min']
resumo_central = df[colunas_numericas].agg(['count', 'mean', 'median', 'std', 'min', 'max']).T
resumo_central

### Contagem de categorias

Para colunas categóricas, value_counts() mostra quantos registros pertencem a cada categoria. Aqui verificamos a situação operacional dos voos.

In [ ]:
df['situacao_voo'].value_counts(dropna=False)

## 2. Schema e qualidade básica

As colunas de horários reais e situações operacionais são rótulos ou informações posteriores. Elas não devem entrar como preditoras no primeiro experimento.

### Dicionário de tipos e valores ausentes

Esta tabela mostra o tipo de cada coluna e quantos valores estão ausentes. Use-a para identificar quais campos podem ser usados com segurança.

In [ ]:
schema = pd.DataFrame({'tipo': df.dtypes.astype(str), 'ausentes': df.isna().sum(), 'percentual_ausente': (100 * df.isna().mean()).round(2)})
schema

### Duplicidades e situações operacionais

Aqui verificamos registros repetidos e observamos quantos voos foram realizados ou cancelados, além das categorias de situação da chegada.

In [ ]:
print('Duplicidades:', int(df.duplicated().sum()))
print('Arquivos de origem:', df['arquivo_origem'].value_counts().sort_index().to_dict())
print('Situação do voo:')
display(df['situacao_voo'].value_counts(dropna=False).to_frame('quantidade'))
print('Situação da chegada:')
display(df['situacao_chegada'].value_counts(dropna=False).to_frame('quantidade'))

## 3. Alvo e distribuição das classes

`cancelado` é uma situação operacional separada. Para previsão de atraso, usamos somente voos realizados com atraso de chegada calculável. O alvo principal é `faixa_atraso`, uma classificação ordinal com seis classes.

### Resumo do alvo e das faixas

Esta célula quantifica cancelamentos, voos realizados, registros elegíveis para modelagem e a distribuição das seis faixas de atraso.

In [ ]:
FAIXAS = {
    0: 'Pontual ou antecipado',
    1: 'Atraso inferior a 15 min',
    2: 'Atraso de 15 a 30 min',
    3: 'Atraso superior a 30 até 45 min',
    4: 'Atraso superior a 45 até 60 min',
    5: 'Atraso superior a 60 min',
}
target_summary = pd.DataFrame({
    'quantidade': [df['cancelado'].sum(), df['realizado'].sum(), len(modelagem)],
}, index=['cancelados', 'realizados', 'registros_elegíveis_para_modelagem'])
distribuicao_faixas = modelagem['faixa_atraso'].value_counts().reindex(FAIXAS.keys(), fill_value=0).rename('quantidade').to_frame()
distribuicao_faixas.index.name = 'codigo'
distribuicao_faixas['classificacao'] = distribuicao_faixas.index.map(FAIXAS)
distribuicao_faixas['percentual'] = (100 * distribuicao_faixas['quantidade'] / len(modelagem)).round(2)
display(target_summary)
display(distribuicao_faixas[['classificacao', 'quantidade', 'percentual']])

### Estatísticas dos atrasos

Calculamos percentis do atraso de partida e de chegada apenas para voos realizados. Os valores negativos representam antecipação.

In [ ]:
realizados = df[df['realizado']].copy()
realizados[['atraso_partida_min', 'atraso_chegada_min']].describe(percentiles=[.01, .05, .25, .5, .75, .95, .99]).round(2)

## 4. Distribuição e outliers

A amostra já revelou registros com atrasos superiores a 24 horas e casos acima de 30 dias. Esses valores não devem ser removidos automaticamente. A classificação em faixas reduz a sensibilidade a diferenças extremas dentro da mesma classe, mas uma regressão em minutos exigiria uma regra de tratamento definida antes da avaliação.

### Investigação de outliers

Esta tabela lista os maiores atrasos de chegada. Observe especialmente os casos acima de 24 horas antes de decidir qualquer tratamento.

In [ ]:
outliers = realizados[realizados['atraso_chegada_min'] > 24 * 60].sort_values('atraso_chegada_min', ascending=False)
print('Atrasos de chegada acima de 24 horas:', len(outliers))
outliers[['arquivo_origem', 'companhia_icao', 'numero_voo', 'origem_icao', 'destino_icao', 'chegada_prevista', 'chegada_real', 'atraso_chegada_min', 'situacao_chegada']].head(20)

### Visualização da distribuição

Os gráficos mostram a distribuição dos atrasos em minutos e a quantidade de registros em cada uma das seis classes ordinais.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
realizados['atraso_chegada_min'].clip(-60, 360).plot.hist(bins=60, ax=axes[0], title='Atraso de chegada (limitado a -60 a 360 min)')
modelagem['faixa_atraso'].value_counts().sort_index().rename(index=FAIXAS).plot.bar(ax=axes[1], title='Distribuição das faixas de atraso')
axes[0].set_xlabel('Minutos')
axes[1].set_xlabel('Faixa de atraso')
plt.tight_layout()

## 5. Padrões por tempo, companhia e aeroporto

Estas tabelas mostram como as seis classes se distribuem ao longo do tempo e entre grupos. São descritivas, não demonstram causalidade e não devem ser usadas diretamente como estimativas do desempenho futuro sem uma divisão temporal.

### Evolução por mês

Resumimos o volume de cada faixa de atraso por mês de referência. A tabela ajuda a identificar mudanças na composição do alvo ao longo do tempo.

In [ ]:
monthly = pd.crosstab(modelagem['mes_referencia'], modelagem['faixa_atraso'])
monthly = monthly.reindex(columns=FAIXAS.keys(), fill_value=0)
monthly = monthly.rename(columns=FAIXAS)
monthly['total_voos'] = monthly.sum(axis=1)
monthly

### Comparação por companhia

Esta tabela compara a quantidade de voos em cada faixa por companhia, considerando companhias com pelo menos 100 voos. Ela é descritiva e não deve ser interpretada como ranking de desempenho.

In [ ]:
by_carrier = pd.crosstab(modelagem['companhia_icao'], modelagem['faixa_atraso'])
by_carrier = by_carrier.reindex(columns=FAIXAS.keys(), fill_value=0).rename(columns=FAIXAS)
by_carrier['total_voos'] = by_carrier.sum(axis=1)
by_carrier = by_carrier.query('total_voos >= 100').sort_values('total_voos', ascending=False)
by_carrier.head(20)

### Comparação por aeroporto e hora

Por fim, observamos a composição das faixas por aeroporto de origem e por hora prevista de partida, sempre como exploração inicial e sem inferir causalidade.

In [ ]:
by_origin = pd.crosstab(modelagem['origem_icao'], modelagem['faixa_atraso'])
by_origin = by_origin.reindex(columns=FAIXAS.keys(), fill_value=0).rename(columns=FAIXAS)
by_origin['total_voos'] = by_origin.sum(axis=1)
by_origin = by_origin.query('total_voos >= 100').sort_values('total_voos', ascending=False)
by_hour = pd.crosstab(modelagem['hora_prevista'], modelagem['faixa_atraso'])
by_hour = by_hour.reindex(columns=FAIXAS.keys(), fill_value=0).rename(columns=FAIXAS)
by_hour['total_voos'] = by_hour.sum(axis=1)
display(by_origin.head(20))
display(by_hour)

## Conclusão provisória

O próximo notebook, `02_modelagem_faixas_atraso.ipynb`, deve usar as variáveis preditoras disponíveis antes do voo, respeitar a janela temporal de treino/validação/teste e comparar os modelos para as seis classes. As tabelas acima são exploratórias e não representam previsão operacional.